In [26]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv')

In [3]:
# data preparation
print(df.isnull().sum())
print(df.info())

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-nul

In [4]:
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
numerical_columns = list(df.dtypes[df.dtypes != 'object'].index)

for c in categorical_columns:
    df[c] = df[c].fillna('NA')

for c in numerical_columns:
    df[c] = df[c].fillna(0)

In [5]:
# question 1 

df['industry'].mode()

0    retail
Name: industry, dtype: object

In [11]:
# question 2
x_var = []
y_var = []
correl = []

for x in df[numerical_columns].corr().index:
    for y in df[numerical_columns].corr().columns:
        if x != y:
            x_var.append(x)
            y_var.append(y)
            correl.append(df[numerical_columns].corr().loc[x, y])
correlation_df = pd.DataFrame({'x': x_var, 'y': y_var, 'correlation': correl})
correlation_df = correlation_df.sort_values(by='correlation', ascending=False)


In [13]:
correlation_df[correlation_df['x'].isin(['interaction_count','number_of_courses_viewed', 'annual_income'])
               & correlation_df['y'].isin(['lead_score','interaction_count', 'interaction_count']) ]

,x,y,correlation
5,annual_income,interaction_count,0.027036
6,annual_income,lead_score,0.015610
10,interaction_count,lead_score,0.009888
2,number_of_courses_viewed,lead_score,-0.004879
1,number_of_courses_viewed,interaction_count,-0.023565


In [14]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [15]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [17]:
# question 3

from sklearn.metrics import mutual_info_score

for c in categorical_columns:
    print(c, round(mutual_info_score(y_train, df_train[c]),2))

lead_source 0.04
industry 0.01
employment_status 0.01
location 0.0


In [19]:
# question 4

from sklearn.feature_extraction import DictVectorizer

from sklearn.linear_model import LogisticRegression

dv = DictVectorizer(sparse=False)
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.5)
print("Accuracy: ",(churn_decision == y_val).mean().round(3))
orig_accuracy = (churn_decision == y_val).mean().round(3)

Accuracy:  0.7


/Users/fdl/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/fdl/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/fdl/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [27]:
# question 5

def model_regression(df_train, y_train, C=1.0):
    dv = DictVectorizer(sparse=False)
    train_dicts = df_train.to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return dv, model

def evaluate_model(dv, model, df_val, y_val):
    
    val_dicts = df_val.to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy = (churn_decision == y_val).mean()
    
    return accuracy

In [30]:
accuracy_results = {}

for column_drop in df_train.columns:
    df_train_temp = df_train.drop(columns=[column_drop])
    dv, model = model_regression(df_train_temp, y_train)
    df_val_temp = df_val.drop(columns=[column_drop])
    accuracy = evaluate_model(dv, model, df_val_temp, y_val)
    accuracy_results[column_drop] = accuracy
    print(f'Dropping {column_drop}, accuracy: {accuracy.round(3)}')

df_accuracy = pd.DataFrame.from_dict(accuracy_results, orient='index', columns=['accuracy'])
df_accuracy['accuracy_diff'] = df_accuracy['accuracy'] - orig_accuracy
df_accuracy = df_accuracy.sort_values(by='accuracy_diff')

df_accuracy[df_accuracy.index.isin(['industry', 'employment_status', 'lead_source'])]


Dropping lead_source, accuracy: 0.703
Dropping industry, accuracy: 0.7
Dropping number_of_courses_viewed, accuracy: 0.556
Dropping annual_income, accuracy: 0.853
Dropping employment_status, accuracy: 0.696
Dropping location, accuracy: 0.71
Dropping interaction_count, accuracy: 0.556
Dropping lead_score, accuracy: 0.706


,accuracy,accuracy_diff
employment_status,0.696246,-0.003754
industry,0.699659,-0.000341
lead_source,0.703072,0.003072


In [31]:
# question 6 

accuracy_results_C = {}
for C in [0.01, 0.1, 1, 10, 100]:
    dv, model = model_regression(df_train, y_train, C=C)
    accuracy = evaluate_model(dv, model, df_val, y_val)
    print(f'C={C}, accuracy: {accuracy}')
    accuracy_results_C[C] = round(accuracy, 3)
    
df_accuracy_C = pd.DataFrame.from_dict(accuracy_results_C, orient='index', columns=['accuracy'])
df_accuracy_C.sort_values(by='accuracy', ascending=False)

C=0.01, accuracy: 0.6996587030716723
C=0.1, accuracy: 0.6996587030716723
C=1, accuracy: 0.6996587030716723
C=10, accuracy: 0.6996587030716723
C=100, accuracy: 0.6996587030716723


,accuracy
0.01,0.7
0.10,0.7
1.00,0.7
10.00,0.7
100.00,0.7
